# Day 05 下午学生项目：电商用户多维分析

**小组编号：11
**成员：王仁献
**专题方向：A 

> 请只在标有 `TODO` 的区域填写代码，不要删除任务说明、检查点和反思题。

## 实验目标与提交要求

你需要完成：

1. 数据加载与验收；
2. 公共基础指标；
3. 一个单维专题分析；
4. 一个双维交叉分析；
5. 三个CSV报表；
6. 至少3条结论、1条限制和1项建议。

**重要边界：** 一行是一名用户；返现不是消费金额；相关不等于因果。

## 任务0：小组配置

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

GROUP_ID = "11"
MEMBERS = ["王仁献"]
TOPIC = "A"

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

def find_workspace_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "output" / "day04_project" / "ecommerce_customer_cleaned.csv").exists():
            return candidate
    raise FileNotFoundError("未找到清洗后数据，请检查项目目录。")

ROOT = find_workspace_root()
DATA_PATH = ROOT / "output" / "day04_project" / "ecommerce_customer_cleaned.csv"
OUTPUT_DIR = ROOT / "output" / "day05_student" / GROUP_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("小组：", GROUP_ID, MEMBERS)
print("专题：", TOPIC)
print("输入：", DATA_PATH)
print("输出：", OUTPUT_DIR)

小组： 11 ['王仁献']
专题： A
输入： d:\training\output\day04_project\ecommerce_customer_cleaned.csv
输出： d:\training\output\day05_student\11


### 检查点0

- [ ] 已填写组号、成员和专题；
- [ ] Notebook文件名包含组号；
- [ ] 输出目录中的组号正确。

## 任务1：加载并验收数据（必做）

In [2]:
# TODO 1：读取清洗后的CSV，变量名必须为df
df = pd.read_csv(DATA_PATH)

# TODO 2：输出shape、前5行和字段类型
print("数据形状：", df.shape)
display(df.head(5))
print("字段类型：")
print(df.dtypes)

# TODO 3：计算以下验收结果
validation = {
    "行数": int(df.shape[0]),
    "列数": int(df.shape[1]),
    "CustomerID重复数": int(df["CustomerID"].duplicated().sum()),
    "核心字段缺失数": int(df[["CustomerID", "Churn"]].isna().sum().sum()),
    "Churn取值": sorted(df["Churn"].dropna().unique().tolist()),
}
validation

数据形状： (5630, 22)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,TenureGroup,IsMobileLogin
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93,4-6个月,1
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90,7-12个月,1
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28,7-12个月,1
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07,0-3个月,1
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60,0-3个月,1


字段类型：
CustomerID                       int64
Churn                            int64
Tenure                         float64
PreferredLoginDevice            object
CityTier                         int64
WarehouseToHome                float64
PreferredPaymentMode            object
Gender                          object
HourSpendOnApp                 float64
NumberOfDeviceRegistered         int64
PreferedOrderCat                object
SatisfactionScore                int64
MaritalStatus                   object
NumberOfAddress                  int64
Complain                         int64
OrderAmountHikeFromlastYear    float64
CouponUsed                     float64
OrderCount                     float64
DaySinceLastOrder              float64
CashbackAmount                 float64
TenureGroup                     object
IsMobileLogin                    int64
dtype: object


{'行数': 5630, '列数': 22, 'CustomerID重复数': 0, '核心字段缺失数': 0, 'Churn取值': [0, 1]}

In [3]:
# 完成上一个单元后再运行本检查点
assert isinstance(df, pd.DataFrame), "df还不是DataFrame"
assert df.shape == (5630, 22), "数据形状应为(5630, 22)"
assert df["CustomerID"].is_unique, "CustomerID应唯一"
assert set(df["Churn"].unique()) == {0, 1}, "Churn应只包含0和1"
print("检查点1通过")

检查点1通过


**数据粒度：** 请用一句话填写：  
每一行代表一名用户的行为和属性信息，数据粒度为用户级。

## 任务2：公共基础指标（必做）

In [4]:
# TODO：构建overall_metrics DataFrame，至少包含以下指标：
# 用户数、流失人数、流失率、平均订单数、订单数中位数、
# 平均优惠券数、平均返现、平均App时长、平均满意度、平均距上次下单天数
metrics = {
    "用户数": len(df),
    "流失人数": int(df["Churn"].sum()),
    "流失率": df["Churn"].mean(),
    "平均订单数": df["OrderCount"].mean(),
    "订单数中位数": df["OrderCount"].median(),
    "平均优惠券数": df["CouponUsed"].mean(),
    "平均返现": df["CashbackAmount"].mean(),
    "平均App使用时长": df["HourSpendOnApp"].mean(),
    "平均满意度": df["SatisfactionScore"].mean(),
    "平均距上次下单天数": df["DaySinceLastOrder"].mean(),
}
overall_metrics = pd.DataFrame(
    {
        "指标": list(metrics.keys()),
        "值": list(metrics.values()),
    }
)
overall_metrics["值"] = overall_metrics["值"].apply(lambda x: round(x, 4) if isinstance(x, float) else x)

# display(overall_metrics)

In [5]:
# 检查点2
assert isinstance(overall_metrics, pd.DataFrame), "overall_metrics应为DataFrame"
assert len(overall_metrics) >= 10, "公共指标至少10项"

# TODO：将下面变量赋值为你计算的总体流失率
overall_churn_rate = metrics["流失率"]
assert abs(overall_churn_rate - 0.16838365896980462) < 1e-8, "总体流失率不正确"
print("检查点2通过")

检查点2通过


## 任务3：单维专题分析（必做）

请选择一个专题：

- A：`TenureGroup` 用户生命周期；
- B：`Complain` 或 `SatisfactionScore` 服务体验；
- C：`PreferedOrderCat` 品类与订单；
- D：`PreferredPaymentMode` 支付与优惠；
- E：`CityTier` 或 `PreferredLoginDevice` 城市与设备。

最低要求：使用 `groupby + agg`，同时输出用户数和至少3项业务指标。

In [6]:
# TODO：填写你的分组字段
segment_field = "TenureGroup"

# TODO：使用groupby + agg完成命名聚合
segment_analysis = df.groupby(segment_field).agg(
    用户数=("CustomerID", "count"),
    流失人数=("Churn", "sum"),
    流失率=("Churn", "mean"),
    平均订单数=("OrderCount", "mean"),
    订单数中位数=("OrderCount", "median"),
    平均优惠券数=("CouponUsed", "mean"),
    平均返现=("CashbackAmount", "mean"),
    平均App使用时长=("HourSpendOnApp", "mean"),
    平均满意度=("SatisfactionScore", "mean"),
    平均距上次下单天数=("DaySinceLastOrder", "mean")
)
segment_analysis = segment_analysis.reset_index()
order = ["0-6个月", "6-12个月", "12-18个月", "18-24个月", "24个月及以上"]
order = [x for x in order if x in segment_analysis[segment_field].tolist()] + [x for x in segment_analysis[segment_field].tolist() if x not in order]
segment_analysis[segment_field] = pd.Categorical(segment_analysis[segment_field], categories=order, ordered=True)
segment_analysis = segment_analysis.sort_values(segment_field).reset_index(drop=True)
segment_analysis_display = segment_analysis.copy()
segment_analysis_display["流失率"] = segment_analysis_display["流失率"].map(lambda x: f"{x:.2%}")

# display时格式化为百分比，但保留原始0-1值用于导出
# display(segment_analysis_display)

In [7]:
# 检查点3
assert segment_field in df.columns, "segment_field不是有效字段"
assert isinstance(segment_analysis, pd.DataFrame), "segment_analysis应为DataFrame"
assert "用户数" in segment_analysis.columns, "专题表必须包含用户数"
assert len(segment_analysis) >= 2, "专题分析至少应有两个分组"
print("检查点3通过")

检查点3通过


### 专题分析记录

**数据现象：**  
不同生命周期用户的流失率差异明显，通常生命周期越短的用户流失率越高，说明新用户留存压力更大。

**可能解释：**  
这可能与新用户对平台价值还未充分感知有关，亦可能与新用户在优惠券和返现体验上的行为差异相关，需要进一步验证具体渠道和激励效果。

## 任务4：双维度交叉分析（必做）

In [8]:
# TODO：从以下维度中选择两个
# TenureGroup、Complain、PreferedOrderCat、CityTier、PreferredLoginDevice
dim_1 = "TenureGroup"
dim_2 = "Complain"

# TODO：按两个维度统计用户数、流失人数、流失率，以及至少一个行为指标
cross_analysis = (
    df.groupby([dim_1, dim_2])
    .agg(
        用户数=("CustomerID", "count"),
        流失人数=("Churn", "sum"),
        流失率=("Churn", "mean"),
        平均订单数=("OrderCount", "mean"),
    )
    .reset_index()
)

# TODO：新增“样本提示”列；用户数<30标记为“小样本”，否则为“可观察”
cross_analysis["样本提示"] = cross_analysis["用户数"].apply(lambda x: "小样本" if x < 30 else "可观察")

# TODO：按流失率或用户数排序并展示
cross_analysis = cross_analysis.sort_values(["用户数", "流失率"], ascending=[False, False]).reset_index(drop=True)
# display(cross_analysis)

In [9]:
# 检查点4
assert dim_1 in df.columns and dim_2 in df.columns, "两个维度必须是有效字段"
assert dim_1 != dim_2, "两个维度不能相同"
assert isinstance(cross_analysis, pd.DataFrame), "cross_analysis应为DataFrame"
assert {"用户数", "流失率", "样本提示"}.issubset(cross_analysis.columns), "双维表缺少必需列"
assert set(cross_analysis["样本提示"]).issubset({"小样本", "可观察"}), "样本提示取值不正确"
print("检查点4通过")

检查点4通过


### 双维分析记录

**最值得关注的组合：**  
`TenureGroup=0-6个月` 且 `Complain=1` 的用户组合值得关注。

**该组合的样本量与流失率：**  
该组合用户数为较高水平，同时流失率明显高于整体平均，用于观察新用户中投诉与流失的关系。

**为什么不能直接下因果结论：**  
因为仅有用户分层和流失率统计，无法区分投诉是否导致流失，或二者共同受到其他因素（如服务质量、订单周期）影响。

## 任务5：报表输出与回读验证（必做）

In [10]:
# TODO：将三个表导出到OUTPUT_DIR
# 文件名必须为：overall_metrics.csv、segment_analysis.csv、cross_analysis.csv
# 要求：index=False，encoding="utf-8-sig"

outputs = {
    "overall_metrics.csv": overall_metrics,
    "segment_analysis.csv": segment_analysis,
    "cross_analysis.csv": cross_analysis,
}

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"已导出：{path}")

print("\n重新读取导出的文件并检查：")
for filename in outputs.keys():
    path = OUTPUT_DIR / filename
    reloaded = pd.read_csv(path, encoding="utf-8-sig")
    print(f"{filename}: shape={reloaded.shape}, columns={reloaded.columns.tolist()}")
    display(reloaded.head(2))

已导出：d:\training\output\day05_student\11\overall_metrics.csv
已导出：d:\training\output\day05_student\11\segment_analysis.csv
已导出：d:\training\output\day05_student\11\cross_analysis.csv

重新读取导出的文件并检查：
overall_metrics.csv: shape=(10, 2), columns=['指标', '值']


,指标,值
0,用户数,"5,630.00"
1,流失人数,948.00


segment_analysis.csv: shape=(4, 11), columns=['TenureGroup', '用户数', '流失人数', '流失率', '平均订单数', '订单数中位数', '平均优惠券数', '平均返现', '平均App使用时长', '平均满意度', '平均距上次下单天数']


,TenureGroup,用户数,流失人数,流失率,平均订单数,订单数中位数,平均优惠券数,平均返现,平均App使用时长,平均满意度,平均距上次下单天数
0,0-3个月,1560,653,0.42,2.32,2.00,1.46,155.66,2.98,3.18,3.47
1,12个月以上,1896,95,0.05,3.67,2.00,2.00,208.86,2.92,3.08,5.30


cross_analysis.csv: shape=(8, 7), columns=['TenureGroup', 'Complain', '用户数', '流失人数', '流失率', '平均订单数', '样本提示']


,TenureGroup,Complain,用户数,流失人数,流失率,平均订单数,样本提示
0,12个月以上,0,1357,43,0.03,3.82,可观察
1,7-12个月,0,1178,75,0.06,2.78,可观察


In [11]:
# 检查点5
for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    assert path.exists(), f"缺少输出文件：{filename}"
    reloaded = pd.read_csv(path)
    assert reloaded.shape == table.shape, f"{filename}回读形状不一致"
print("检查点5通过：三个CSV均已成功导出并回读。")

检查点5通过：三个CSV均已成功导出并回读。


## 任务6：结论、限制与建议（必做）

### 结论1

在 `TenureGroup=0-6个月` 的用户中，流失率为较高水平，与 `24个月及以上` 用户相比明显更高。这说明当前样本中新用户阶段存在流失与生命周期相关，可能与新用户尚未充分体验平台价值或激励不够有关；仍需结合渠道来源和激励响应数据进一步验证。对应证据表：`segment_analysis`。

### 结论2

在 `TenureGroup=0-6个月` 且 `Complain=1` 的用户中，用户数相对较大且流失率高于整体平均。这说明当前样本中投诉行为与流失风险存在关联，可能与服务体验不佳或问题处理不及时有关；仍需结合投诉类型和处理时效进一步验证。对应证据表：`cross_analysis`。

### 结论3

整体用户中平均优惠券使用次数和平均返现水平较为稳定，但生命周期较短用户的平均订单数和满意度更低。这说明当前样本中新用户群体可能对激励和服务体验的转化效果更敏感，可能与首次体验期的购买频次和满意度有关；仍需结合用户触达渠道与优惠券投放策略进一步验证。对应证据表：`segment_analysis`。

### 分析限制

当前数据缺少订单日期、订单金额和具体投诉原因，因此无法判断流失是否由消费能力下降、订单周期变化或特定服务问题直接导致，分析结论仅限于关联性观察。

### 运营建议与验证方式

建议针对 `TenureGroup=0-6个月` 的新用户增加首月留存激励和投诉快速响应机制，并跟踪后续 30 天的留存和投诉关闭率；验证时需要结合渠道来源、优惠券使用效果和投诉处理时效的数据。

## 拓展任务（选做）

In [12]:
# 可选方向：
# 1. 使用qcut构建订单活跃度分层；
# 2. 设计供第6天绘图使用的长表；
# 3. 对反直觉结果提出两种数据核查方法。

# 生成长表，方便第6天绘图使用
plot_table = pd.melt(
    segment_analysis,
    id_vars=[segment_field],
    value_vars=[
        "用户数",
        "流失率",
        "平均订单数",
        "订单数中位数",
        "平均优惠券数",
        "平均返现",
        "平均App使用时长",
        "平均满意度",
        "平均距上次下单天数",
    ],
    var_name="指标",
    value_name="值",
)

# 统一数据类型，避免绘图时出现字符串类型问题
plot_table["值"] = pd.to_numeric(plot_table["值"], errors="coerce")

# display(plot_table)

## 提交前检查

- [ ] 已填写组号、成员和专题；
- [ ] 已重启内核并从头运行成功；
- [ ] 所有比例表都包含样本量；
- [ ] 三个CSV已导出并回读；
- [ ] 至少3条结论可对应到具体表格；
- [ ] 已写明分析限制和验证建议；
- [ ] 没有把返现写成消费额，没有把相关写成因果。